In [ ]:
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel

# 1. 加载环境变量
load_dotenv()

# 2. 配置模型 (确保环境变量中有 OPENAI_BASE_URL 和 OPENAI_API_KEY)
model = OpenAIChatModel(
    model_name='qwen-plus',
)

In [ ]:
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext

@dataclass
class UserDatabase:
    """模拟一个存储用户偏好的数据库"""
    user_id: str
    username: str
    last_topic: str

# 定义 Agent，并指定 deps 的类型
agent = Agent(
    'qwen-plus',
    deps_type=UserDatabase,
    system_prompt='你是一个贴心的助手。'
)

# 使用装饰器定义一个工具，让 Agent 自动去“查阅”记忆
@agent.system_prompt
def add_user_context(ctx: RunContext[UserDatabase]) -> str:
    return f"当前对话用户是 {ctx.deps.username}，他上次关心的内容是 {ctx.deps.last_topic}。"

async def main():
    # 模拟从数据库取出的数据
    my_db = UserDatabase(user_id="123", username="老王", last_topic="Python 异步编程")

    # 运行时注入 deps
    result = await agent.run('根据我的兴趣推荐一个今天的学习计划', deps=my_db)
    print(result.data)

if __name__ == '__main__':
    import asyncio
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop